In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import Wav2Vec2Model, Wav2Vec2Config
# 
# class TextIndependentPhoneAligner(nn.Module):
#     def __init__(self, num_phone_classes=39):
#         super().__init__()
# 
#         # Use pretrained Wav2Vec2 as feature extractor
#         self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
# 
#         # Freeze base Wav2Vec2 weights
#         for param in self.wav2vec.parameters():
#             param.requires_grad = False
# 
#         # Additional layers for frame-wise phone classification
#         self.frame_classifier = nn.Sequential(
#             nn.Linear(768, 512),  # Wav2Vec2 hidden size is 768
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(512, num_phone_classes)
#         )
# 
#     def forward(self, audio_input):
#         """
#         Perform text-independent phone alignment
#         
#         Args:
#             audio_input (torch.Tensor): Raw audio waveform
#         
#         Returns:
#             phone_predictions (torch.Tensor): Frame-wise phone class probabilities
#         """
#         # Extract features from Wav2Vec2
#         wav2vec_output = self.wav2vec(audio_input)
#         hidden_states = wav2vec_output.last_hidden_state
# 
#         # Classify each frame into phone classes
#         phone_predictions = self.frame_classifier(hidden_states)
# 
#         return phone_predictions
# 
#     def get_phone_boundaries(self, phone_predictions):
#         """
#         Extract phone boundaries from frame-wise predictions
#         
#         Args:
#             phone_predictions (torch.Tensor): Frame-wise phone probabilities
#         
#         Returns:
#             boundaries (list): Phone segment boundaries
#         """
#         # Get most likely phone class for each frame
#         frame_classes = torch.argmax(phone_predictions, dim=-1)
# 
#         # Find boundary transitions
#         boundaries = []
#         current_phone = frame_classes[0]
#         start_frame = 0
# 
#         for frame_idx, phone_class in enumerate(frame_classes[1:], 1):
#             if phone_class != current_phone:
#                 boundaries.append({
#                     'start': start_frame,
#                     'end': frame_idx,
#                     'phone_class': current_phone
#                 })
#                 start_frame = frame_idx
#                 current_phone = phone_class
# 
#         # Add final segment
#         boundaries.append({
#             'start': start_frame,
#             'end': len(frame_classes),
#             'phone_class': current_phone
#         })
# 
#         return boundaries
# 
# class TextIndependentPhoneAlignmentTrainer:
#     def __init__(self, num_phone_classes=39):
#         self.model = TextIndependentPhoneAligner(num_phone_classes)
#         self.optimizer = torch.optim.Adam(
#             self.model.frame_classifier.parameters(),
#             lr=1e-4
#         )
# 
#     def train_step(self, audio_batch, phone_labels):
#         """
#         Single training step for frame-wise phone classification
#         
#         Args:
#             audio_batch (torch.Tensor): Batch of audio waveforms
#             phone_labels (torch.Tensor): Frame-wise phone labels
#         """
#         self.optimizer.zero_grad()
# 
#         # Forward pass
#         phone_predictions = self.model(audio_batch)
# 
#         # Cross-entropy loss for frame classification
#         loss = F.cross_entropy(
#             phone_predictions.view(-1, phone_predictions.size(-1)),
#             phone_labels.view(-1)
#         )
# 
#         # Backpropagation
#         loss.backward()
#         self.optimizer.step()
# 
#         return loss.item()
# 
# # Example usage
# def main():
#     # Hyperparameters
#     num_phone_classes = 39
#     batch_size = 32
# 
#     # Initialize model and trainer
#     trainer = TextIndependentPhoneAlignmentTrainer(num_phone_classes)
# 
#     # Training loop (pseudo-code)
#     for epoch in range(10):
#         for batch in data_loader:
#             audio_batch, phone_labels = batch
#             loss = trainer.train_step(audio_batch, phone_labels)
#             print(f"Epoch {epoch}, Loss: {loss}")
# 
#     # Inference
#     test_audio = torch.randn(1, 16000)  # 1 second of audio
#     phone_predictions = trainer.model(test_audio)
#     phone_boundaries = trainer.model.get_phone_boundaries(phone_predictions)
# 
#     print("Phone Boundaries:", phone_boundaries)
# 
# if __name__ == "__main__":
#     main()


In [1]:
import os
import torch
import torchaudio
import numpy as np
from transformers import Wav2Vec2Model, Wav2Vec2Config
import torch.nn as nn
import torch.nn.functional as F

class JapaneseSyllableAligner(nn.Module):
    def __init__(self,
                 syllable_tokens,
                 wav2vec_model_name='facebook/wav2vec2-base',
                 feature_extraction_method='resample'):
        """
        Initialize Japanese Syllable Alignment Model
        
        Args:
            syllable_tokens (list): List of Japanese syllable tokens
            wav2vec_model_name (str): Pretrained Wav2Vec2 model
            feature_extraction_method (str): Method for audio preprocessing
        """
        super().__init__()

        # Configure Wav2Vec2 with custom settings for Japanese
        config = Wav2Vec2Config.from_pretrained(wav2vec_model_name)
        config.update({
            # Japanese-specific configurations
            'hidden_size': 768,  # Default, but can be adjusted
            'num_attention_heads': 12,
            'num_hidden_layers': 12,
            'layer_norm_eps': 1e-5,
            'feat_extract_activation': 'gelu',

            # Audio preprocessing
            'feat_extract_norm': 'layer',
            'conv_bias': True,
            'conv_dropout': 0.1,
            'activation_dropout': 0.1,
            'attention_dropout': 0.1,
        })

        # Load Wav2Vec2 with custom config
        self.wav2vec = Wav2Vec2Model.from_pretrained(
            wav2vec_model_name,
            config=config
        )

        # Freeze base Wav2Vec2 weights
        for param in self.wav2vec.parameters():
            param.requires_grad = False

        # Syllable-specific classification head
        self.syllable_classifier = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, len(syllable_tokens))
        )

        # Audio resampling setup
        self.feature_extraction_method = feature_extraction_method
        self.target_sample_rate = 16000  # Standard for Wav2Vec2

    def preprocess_audio(self, audio_path):
        """
        Preprocess audio file for model input
        
        Args:
            audio_path (str): Path to audio file
        
        Returns:
            torch.Tensor: Preprocessed audio tensor
        """
        # Load audio with torchaudio
        waveform, original_sample_rate = torchaudio.load(audio_path)

        # Resample if needed
        if original_sample_rate != self.target_sample_rate:
            resampler = torchaudio.transforms.Resample(
                orig_freq=original_sample_rate,
                new_freq=self.target_sample_rate
            )
            waveform = resampler(waveform)

        # Ensure mono channel
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        return waveform.squeeze()

    def forward(self, audio_input):
        """
        Perform syllable prediction
        
        Args:
            audio_input (torch.Tensor): Preprocessed audio tensor
        
        Returns:
            torch.Tensor: Syllable prediction probabilities
        """
        # Extract features from Wav2Vec2
        wav2vec_output = self.wav2vec(audio_input.unsqueeze(0))
        hidden_states = wav2vec_output.last_hidden_state

        # Classify syllables
        syllable_predictions = self.syllable_classifier(hidden_states)

        return syllable_predictions.squeeze()

    def predict_syllable_boundaries(self, audio_path, syllable_tokens):
        """
        Predict syllable boundaries for a given audio file
        
        Args:
            audio_path (str): Path to audio file
            syllable_tokens (list): List of syllable tokens
        
        Returns:
            list: Predicted syllable boundaries
        """
        # Preprocess audio
        audio_input = self.preprocess_audio(audio_path)

        # Get syllable predictions
        with torch.no_grad():
            syllable_predictions = self(audio_input)

        # Extract boundaries
        predicted_syllables = torch.argmax(syllable_predictions, dim=-1)

        # Boundary extraction logic
        boundaries = []
        current_syllable = predicted_syllables[0]
        start_frame = 0

        for frame_idx, syllable in enumerate(predicted_syllables[1:], 1):
            if syllable != current_syllable:
                boundaries.append({
                    'start_frame': start_frame,
                    'end_frame': frame_idx,
                    'syllable': syllable_tokens[current_syllable]
                })
                start_frame = frame_idx
                current_syllable = syllable

        # Add final syllable
        boundaries.append({
            'start_frame': start_frame,
            'end_frame': len(predicted_syllables),
            'syllable': syllable_tokens[current_syllable]
        })

        return boundaries

class JapaneseSyllableAlignmentTrainer:
    def __init__(self, syllable_tokens, learning_rate=1e-4):
        self.model = JapaneseSyllableAligner(syllable_tokens)
        self.optimizer = torch.optim.Adam(
            self.model.syllable_classifier.parameters(),
            lr=learning_rate
        )
        self.syllable_tokens = syllable_tokens

    def train_batch(self, audio_paths, syllable_labels):
        """
        Train on a batch of audio files
        
        Args:
            audio_paths (list): List of audio file paths
            syllable_labels (torch.Tensor): Corresponding syllable labels
        
        Returns:
            float: Training loss
        """
        self.optimizer.zero_grad()

        # Preprocess audio files
        audio_inputs = [self.model.preprocess_audio(path) for path in audio_paths]
        audio_inputs = torch.stack(audio_inputs)

        # Forward pass
        syllable_predictions = self.model(audio_inputs)

        # Compute loss
        loss = F.cross_entropy(
            syllable_predictions.view(-1, len(self.syllable_tokens)),
            syllable_labels.view(-1)
        )

        # Backpropagate
        loss.backward()
        self.optimizer.step()

        return loss.item()

def process_japanese_audio_directory(
        audio_directory,
        syllable_tokens,
        output_directory='./syllable_alignments'
):
    """
    Process all Japanese audio files in a directory
    
    Args:
        audio_directory (str): Directory containing Japanese audio files
        syllable_tokens (list): List of Japanese syllable tokens
        output_directory (str): Directory to save alignment results
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_directory, exist_ok=True)

    # Initialize model
    aligner = JapaneseSyllableAligner(syllable_tokens)

    # Process each audio file
    for audio_file in os.listdir(audio_directory):
        if audio_file.endswith(('.mp3', '.wav', '.flac')):
            full_path = os.path.join(audio_directory, audio_file)

            try:
                # Predict syllable boundaries
                boundaries = aligner.predict_syllable_boundaries(
                    full_path,
                    syllable_tokens
                )

                # Save results
                output_file = os.path.join(
                    output_directory,
                    f"{os.path.splitext(audio_file)[0]}_syllables.json"
                )

                # You might want to use json to save the boundaries
                import json
                with open(output_file, 'w', encoding='utf-8') as f:
                    json.dump(boundaries, f, ensure_ascii=False, indent=2)

                print(f"Processed {audio_file}")

            except Exception as e:
                print(f"Error processing {audio_file}: {e}")

# Example usage
def main():
    # Japanese syllable tokens (example set, replace with your actual tokens)
    japanese_syllables = [
        'あ', 'い', 'う', 'え', 'お',
        'か', 'き', 'く', 'け', 'こ',
        'さ', 'し', 'す', 'せ', 'そ',
        'た', 'ち', 'つ', 'て', 'と',
        'な', 'に', 'ぬ', 'ね', 'の',
        'は', 'ひ', 'ふ', 'へ', 'ほ',
        'ま', 'み', 'む', 'め', 'も',
        'や', 'ゆ', 'よ',
        'ら', 'り', 'る', 'れ', 'ろ',
        'わ', 'を', 'ん',
        'が', 'ぎ', 'ぐ', 'げ', 'ご',
        'ざ', 'じ', 'ず', 'ぜ', 'ぞ',
        'だ', 'ぢ', 'づ', 'で', 'ど',
        'ば', 'び', 'ぶ', 'べ', 'ぼ',
        'ぱ', 'ぴ', 'ぷ', 'ぺ', 'ぽ',
        'きゃ', 'きゅ', 'きょ',
        'しゃ', 'しゅ', 'しょ',
        'ちゃ', 'ちゅ', 'ちょ',
        'にゃ', 'にゅ', 'にょ',
        'ひゃ', 'ひゅ', 'ひょ',
        'みゃ', 'みゅ', 'みょ',
        'りゃ', 'りゅ', 'りょ',
        'ぎゃ', 'ぎゅ', 'ぎょ',
        'じゃ', 'じゅ', 'じょ',
        'びゃ', 'びゅ', 'びょ',
        'ぴゃ', 'ぴゅ', 'ぴょ',
        'でぃ', 'ふぁ', 'ふぃ', 'ふぇ', 'ふぉ',
    ]

    # Directory with Japanese audio files
    audio_directory = './japanese_audio'

    # Process the directory
    process_japanese_audio_directory(
        audio_directory,
        japanese_syllables
    )

if __name__ == "__main__":
    main()


/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-12-11 20:53:28.439042: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-11 20:53:28.478743: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-12-11 20:53:28.478758: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
/home/eizigi/.local/sh

FileNotFoundError: [Errno 2] No such file or directory: './japanese_audio'